# SOLUTIONS: Interactive example of non numpy-able code optimization

In the debugging lecture last year we showed an example of a [not working] code to do the following:

* Consider a sequence of numbers a0, a1, ..., an, in which an element is equal to the sum of squared digits of the previous element. The sequence ends once an element that has already been in the sequence appears again. So that an = a0. Given the first element a0, find the length of the sequence.

So for example if a0 = 16. `1**2 + 6**2 = 37` -> `3**2 + 7**2 = 58` -> `5**2 + 8**2 = 89` -> `8**2 + 9**2 = 145` -> `1**2 + 4**2 + 5**2 = 42` -> `4**2 + 2**2 = 20` -> `2**2 + 0**2 = 4` -> `4**2 = 16` We've already seen 16 so we stop here. The sequence is `[16,37,58,89,145,42,20,4,16]` which has a length of 9.


The code that we used had some optimisations, but was not particularly clear. To try to understand it better let's write this out in a code that approaches the problem in a different way, with comments etc. to try to make it clearer what is going on


In [1]:
def square_digits(input_number):

    # Initialize by setting the list of outputs equal to the input
    output_list = [input_number]
    # And setting the last_number variable to the input
    last_number = input_number

    # This is basically a for-loop that will only exit when we explicitly say "exit"
    while 1:
        # Step one: We must identify the digits of last_number
        # Cast to a string
        last_number = str(last_number)
        # And then convert to a list of integers. We do this using a list comprehension, which is powerful, but not fast
        digits = [int(digit) for digit in last_number]
        # So if last_number is 49120 then digits = [4,9,1,2,0]

        # We can then sum the digits squared using another list comprehension
        # digits = [4,9,1,2,0] -> 16 + 81 + 1 + 4 = 102
        digit_squared_sum = sum([digit*digit for digit in digits])

        # Is this value already in the list?
        if digit_squared_sum in output_list:
            # Add this value and then exist
            output_list.append(digit_squared_sum)
            break
        else:
            # Else add the value and then continue
            output_list.append(digit_squared_sum)
            last_number = digit_squared_sum

    # Uncomment the line below if you want to see the full list. The logging module would be better for this
    # in "production" code, but as an example print is fine!
    #
    # print(output_list)

    return len(output_list)

print("square_digits(103) gives:",square_digits(103), "\n Should be 4")
print()
print("square_digits(612) gives:",square_digits(612), "\n Should be 16")
print()


square_digits(103) gives: 4 
 Should be 4

square_digits(612) gives: 16 
 Should be 16



In [2]:
# Let's try it with the following
very_long_integer = 2**100
square_digits(very_long_integer)


13

Using the profiling tools demonstrated in notebook 3:

* Use timeit to calculate how long the function takes to run with the `very_long_integer`
* Use lprun to determine how long the code spends at each line

In [3]:
# Timing function
%timeit square_digits(very_long_integer)

16.4 µs ± 3.42 µs per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [5]:
import sys
!{sys.executable} -m pip install line_profiler
!{sys.executable} -m pip install memory_profiler

%load_ext line_profiler
%load_ext memory_profiler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.9 MB/s eta 0:00:00


In [8]:
# Profile the function
%lprun -T lprof0 -f square_digits square_digits(very_long_integer)

print(open('lprof0', 'r').read())


*** Profile printout saved to text file 'lprof0'.
Timer unit: 1e-09 s

Total time: 5.995e-05 s
File: /tmp/ipykernel_1107/327163771.py
Function: square_digits at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def square_digits(input_number):
     2                                           
     3                                               # Initialize by setting the list of outputs equal to the input
     4         1       1115.0   1115.0      1.9      output_list = [input_number]
     5                                               # And setting the last_number variable to the input
     6         1        458.0    458.0      0.8      last_number = input_number
     7                                           
     8                                               # This is basically a for-loop that will only exit when we explicitly say "exit"
     9        12       4242.0    353.5      7.1      while 1:
    10

Here is the optimized code that we used last year (with the bug fixed). As before run timeit and lprof on this code:

In [9]:
def square_digits_withoutstr(input_number):

    cur = input_number
    was = set()

    while not (cur in was):
        was.add(cur)
        nxt = 0
        while cur > 0:
            nxt += (cur % 10) * (cur % 10)
            cur //= 10
        cur = nxt

    return len(was) + 1


In [12]:
# Run timeit and lprun here
%timeit square_digits_withoutstr(16)

2.25 µs ± 55.2 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [13]:
%lprun -T lprof0 -f square_digits_withoutstr square_digits_withoutstr(16)

print(open('lprof0', 'r').read())


*** Profile printout saved to text file 'lprof0'.
Timer unit: 1e-09 s

Total time: 7.0749e-05 s
File: /tmp/ipykernel_1107/2076506614.py
Function: square_digits_withoutstr at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def square_digits_withoutstr(input_number):
     2                                           
     3         1       1674.0   1674.0      2.4      cur = input_number
     4         1       1437.0   1437.0      2.0      was = set()
     5                                           
     6         9       5316.0    590.7      7.5      while not (cur in was):
     7         8       5742.0    717.8      8.1          was.add(cur)
     8         8       3705.0    463.1      5.2          nxt = 0
     9        24      11539.0    480.8     16.3          while cur > 0:
    10        16      25494.0   1593.4     36.0              nxt += (cur % 10) * (cur % 10)
    11        16       8290.0    518.1     11.7 

Now repeat the process using:
```very_long_integer=2**100000```

In [14]:
# Run timeit and lprun here

very_long_integer = 2**100000

In [15]:
%timeit square_digits_withoutstr(very_long_integer)

605 ms ± 8.32 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
%lprun -T lprof0 -f square_digits_withoutstr square_digits_withoutstr(very_long_integer)

print(open('lprof0', 'r').read())


*** Profile printout saved to text file 'lprof0'.
Timer unit: 1e-09 s

Total time: 0.637214 s
File: /tmp/ipykernel_1107/2076506614.py
Function: square_digits_withoutstr at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def square_digits_withoutstr(input_number):
     2                                           
     3         1        920.0    920.0      0.0      cur = input_number
     4         1        820.0    820.0      0.0      was = set()
     5                                           
     6        19      16595.0    873.4      0.0      while not (cur in was):
     7        18      16031.0    890.6      0.0          was.add(cur)
     8        18       5984.0    332.4      0.0          nxt = 0
     9     30160   15410866.0    511.0      2.4          while cur > 0:
    10     30142  405039944.0  13437.7     63.6              nxt += (cur % 10) * (cur % 10)
    11     30142  216709591.0   7189.6     34.0   

##  Summary

Think about what these results are telling you. Some things to highlight:

* You are learning how to identify which parts of a function you need to think about when optimising. Never bother optimising any part of your code that is not taking a significant fraction of the total time.
* The optimal solution to a problem can depend on the input. For shorter inputs, converting an integer to a string and then back to a list of integers adds an overhead that is the dominant computational cost. However, as the input integers become very large, operations like dividing by 10 (which for a computer is not as trivial as it seems, because computers think in binary numbers, not base-10 numbers) become expensive (look up how python handles big integers if you want to understand this better). In this case the overhead of converting to a string is faster.
* Although the second case was faster for normal-sized integers, it doesn't seem that using it would really be a good decision if the code is more difficult to parse.